# 🐾 Animal Visual Search — Clean Benchmark v1

**Môn:** Học máy  
**Mục tiêu:** tạo benchmark sạch và công bằng cho bài toán **animal classification + image retrieval + cross-dataset generalization**.

> Notebook này thay thế benchmark sơ bộ trước đó. Các metric cũ được xem là **preliminary**, vì audit đã phát hiện ảnh có perceptual hash trùng giữa Train ↔ Validation/Test.

## Nguyên tắc của Benchmark v1

1. Audit ảnh **trước khi split**.
2. Phát hiện exact duplicate bằng **SHA256**.
3. Phát hiện near-duplicate bằng **dHash + pHash** trong cùng species.
4. Giữ **một representative cho mỗi duplicate cluster**, rồi mới split.
5. Tất cả model dùng **cùng Train / Validation / Test cố định**.
6. Cross-dataset benchmark được audit duplicate giữa **Animal-90 ↔ Wild Animals** trước khi đánh giá.
7. Báo cáo cả:
   - Classification: Accuracy, Macro-F1
   - Retrieval: Precision@5, mAP@10
   - Cross-dataset retrieval: Precision@5, mAP@10

## Model được benchmark

- PCA + KNN
- HOG + Linear SVM
- ResNet18 fine-tuned
- DINOv2 frozen embeddings + Linear SVM / FAISS

### Chế độ chạy

Mặc định `QUICK_MODE = True` để xác nhận pipeline.  
Khi chạy kết quả chính thức, đổi thành:

```python
QUICK_MODE = False
```

Kết quả báo cáo cuối kỳ chỉ nên lấy từ **Full mode**.

## 0. Cài thư viện

In [ ]:
!pip -q install kagglehub scikit-learn scikit-image imagehash faiss-cpu pandas matplotlib pillow tqdm

## 1. Import & cấu hình

In [ ]:
import os
import gc
import math
import random
import hashlib
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score

from skimage.feature import hog
import imagehash
import faiss

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import kagglehub

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# -----------------------------
# RUN MODE
# -----------------------------
QUICK_MODE = True

QUICK_NUM_CLASSES = 20
QUICK_MAX_IMAGES_PER_CLASS = 40

IMG_SIZE_BASELINE = 64
IMG_SIZE_HOG = 128
IMG_SIZE_CNN = 224

BATCH_SIZE = 32 if DEVICE == "cuda" else 16
# 0 ổn định hơn trên Colab/Python mới, tránh multiprocessing cleanup errors.
NUM_WORKERS = 0

EPOCHS = 2 if QUICK_MODE else 8
LR = 3e-4
TOP_K = 5

# Conservative near-duplicate thresholds.
DHASH_THRESHOLD = 4
PHASH_THRESHOLD = 6

OUTPUT_DIR = Path("/content/animal_visual_search_clean_v1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "QUICK_MODE": QUICK_MODE,
    "EPOCHS": EPOCHS,
    "DHASH_THRESHOLD": DHASH_THRESHOLD,
    "PHASH_THRESHOLD": PHASH_THRESHOLD,
})

## 2. Tải dữ liệu

In [ ]:
DATASET_A_SLUG = "iamsouravbanerjee/animal-image-dataset-90-different-animals"
DATASET_B_SLUG = "whenamancodes/wild-animals-images"

DATASET_A_PATH = Path(kagglehub.dataset_download(DATASET_A_SLUG))
DATASET_B_PATH = Path(kagglehub.dataset_download(DATASET_B_SLUG))

print("Animal-90:", DATASET_A_PATH)
print("Wild Animals:", DATASET_B_PATH)

## 3. Tìm cấu trúc ImageFolder và tạo metadata

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".jfif"}

def is_image(path: Path):
    return path.suffix.lower() in IMAGE_EXTS

def find_best_imagefolder_root(root: Path):
    root = Path(root)
    candidates = []

    for d in [root] + [p for p in root.rglob("*") if p.is_dir()]:
        children = [c for c in d.iterdir() if c.is_dir()]
        class_like = []
        total_imgs = 0

        for c in children:
            n = sum(
                1 for p in c.rglob("*")
                if p.is_file() and is_image(p)
            )
            if n > 0:
                class_like.append((c, n))
                total_imgs += n

        if len(class_like) >= 2:
            candidates.append((len(class_like), total_imgs, d))

    if not candidates:
        raise RuntimeError(f"Không tìm được ImageFolder root trong {root}")

    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return candidates[0][2]

def normalize_species_name(name: str):
    name = name.lower().strip().replace("_", "-").replace(" ", "-")
    for suffix in [
        "-resize-224", "-resize-300", "-resize-512",
        "-224", "-300", "-512"
    ]:
        if name.endswith(suffix):
            name = name[:-len(suffix)]
    return name.strip("-")

def build_metadata_from_root(root: Path, dataset_name: str):
    rows = []

    for class_dir in sorted([p for p in Path(root).iterdir() if p.is_dir()]):
        imgs = [
            p for p in class_dir.rglob("*")
            if p.is_file() and is_image(p)
        ]
        if not imgs:
            continue

        raw_class = class_dir.name
        species = normalize_species_name(raw_class)

        for p in imgs:
            rows.append({
                "dataset": dataset_name,
                "path": str(p),
                "raw_class": raw_class,
                "species": species,
            })

    return pd.DataFrame(rows)

ROOT_A = find_best_imagefolder_root(DATASET_A_PATH)
ROOT_B = find_best_imagefolder_root(DATASET_B_PATH)

meta_a_raw = build_metadata_from_root(ROOT_A, "animal90")
meta_b_raw = build_metadata_from_root(ROOT_B, "wild_animals")

# Wild Animals thường cung cấp cùng collection ở 224/300/512.
# Chỉ giữ 224 nếu có.
raw_lower = meta_b_raw["raw_class"].str.lower()
if raw_lower.str.contains("224").any():
    meta_b_raw = meta_b_raw[raw_lower.str.contains("224")].copy().reset_index(drop=True)

print("Animal-90 root:", ROOT_A)
print("Wild Animals root:", ROOT_B)
print("Animal-90 raw:", meta_a_raw.shape)
print("Wild Animals raw (224 only if available):", meta_b_raw.shape)

## 4. Audit file integrity + SHA256 + perceptual hashes

Ta tính ba dấu vết:

- `sha256`: exact duplicate
- `dhash`: cấu trúc cường độ cục bộ
- `phash`: perceptual frequency structure

Near-duplicate chỉ được cluster khi **cùng species** và đồng thời thỏa cả hai ngưỡng dHash/pHash. Đây là lựa chọn bảo thủ để hạn chế xoá nhầm hai ảnh khác nhau nhưng nhìn tương tự.

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def audit_and_hash(df, desc="Audit"):
    rows = []
    bad = []

    for row in tqdm(df.to_dict("records"), desc=desc):
        path = row["path"]
        try:
            with Image.open(path) as img:
                rgb = img.convert("RGB")
                # Force image decoding.
                rgb.load()
                dh = str(imagehash.dhash(rgb, hash_size=8))
                ph = str(imagehash.phash(rgb, hash_size=8))

            rows.append({
                **row,
                "sha256": sha256_file(path),
                "dhash": dh,
                "phash": ph,
            })
        except Exception as e:
            bad.append({
                "path": path,
                "dataset": row.get("dataset"),
                "species": row.get("species"),
                "error": repr(e),
            })

    return pd.DataFrame(rows), pd.DataFrame(bad)

meta_a, bad_a = audit_and_hash(meta_a_raw, "Audit Animal-90")
meta_b, bad_b = audit_and_hash(meta_b_raw, "Audit Wild Animals")

print("Animal-90 valid/corrupt:", len(meta_a), len(bad_a))
print("Wild Animals valid/corrupt:", len(meta_b), len(bad_b))

if len(bad_a):
    display(bad_a.head())
if len(bad_b):
    display(bad_b.head())

## 5. Exact duplicate audit

In [ ]:
def exact_duplicate_summary(df, name):
    dup = df[df.duplicated("sha256", keep=False)].copy()
    groups = dup.groupby("sha256") if len(dup) else []
    print(f"=== {name}: Exact duplicate audit ===")
    print("Rows in duplicate groups:", len(dup))
    print("Duplicate groups:", dup["sha256"].nunique() if len(dup) else 0)

    if len(dup):
        cross_label = (
            dup.groupby("sha256")["species"]
               .nunique()
               .sort_values(ascending=False)
        )
        print("Exact duplicate groups spanning >1 label:",
              int((cross_label > 1).sum()))
    return dup

exact_a = exact_duplicate_summary(meta_a, "Animal-90")
exact_b = exact_duplicate_summary(meta_b, "Wild Animals")

## 6. Tạo near-duplicate clusters **trước khi split**

In [ ]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1

def hamming_hex(a, b):
    return (int(a, 16) ^ int(b, 16)).bit_count()

def assign_duplicate_groups(
    df,
    dhash_threshold=DHASH_THRESHOLD,
    phash_threshold=PHASH_THRESHOLD
):
    out = df.reset_index(drop=True).copy()
    dsu = DSU(len(out))

    # 1) Exact duplicate: global, bất kể label.
    sha_groups = out.groupby("sha256").indices
    for _, idxs in sha_groups.items():
        idxs = list(idxs)
        for j in range(1, len(idxs)):
            dsu.union(idxs[0], idxs[j])

    # 2) Conservative perceptual near-duplicate: only within same species.
    for species, idxs in tqdm(
        out.groupby("species").indices.items(),
        desc="Near-duplicate clustering"
    ):
        idxs = list(idxs)
        for i in range(len(idxs)):
            ai = idxs[i]
            dh_a = out.at[ai, "dhash"]
            ph_a = out.at[ai, "phash"]

            for j in range(i + 1, len(idxs)):
                bi = idxs[j]
                dh_b = out.at[bi, "dhash"]
                ph_b = out.at[bi, "phash"]

                if (
                    hamming_hex(dh_a, dh_b) <= dhash_threshold
                    and
                    hamming_hex(ph_a, ph_b) <= phash_threshold
                ):
                    dsu.union(ai, bi)

    root_to_gid = {}
    gids = []
    next_gid = 0

    for i in range(len(out)):
        root = dsu.find(i)
        if root not in root_to_gid:
            root_to_gid[root] = next_gid
            next_gid += 1
        gids.append(root_to_gid[root])

    out["dup_group"] = gids
    group_sizes = out.groupby("dup_group").size()
    out["dup_group_size"] = out["dup_group"].map(group_sizes)

    return out

meta_a_grouped = assign_duplicate_groups(meta_a)

group_sizes = meta_a_grouped.groupby("dup_group").size().sort_values(ascending=False)
print("Images:", len(meta_a_grouped))
print("Duplicate clusters (>1 image):", int((group_sizes > 1).sum()))
print("Images participating in duplicate clusters:",
      int(group_sizes[group_sizes > 1].sum()))
display(group_sizes.head(15).rename("cluster_size").to_frame())

## 7. Deduplicate rồi mới chọn working set

Mỗi duplicate cluster giữ **một ảnh đại diện**. Sau bước này, cùng một nội dung thị giác không thể bị rải sang Train/Val/Test.

In [ ]:
# Nếu một duplicate cluster chứa >1 species, đây là label conflict.
# Không chọn ngẫu nhiên một label: loại toàn bộ cluster khỏi benchmark và lưu để audit.
group_label_counts = meta_a_grouped.groupby("dup_group")["species"].nunique()
ambiguous_dup_groups = set(
    group_label_counts[group_label_counts > 1].index.tolist()
)

ambiguous_duplicates = meta_a_grouped[
    meta_a_grouped["dup_group"].isin(ambiguous_dup_groups)
].copy()

meta_a_unambiguous = meta_a_grouped[
    ~meta_a_grouped["dup_group"].isin(ambiguous_dup_groups)
].copy()

# Deterministic representative: sort path rồi giữ phần tử đầu mỗi cluster.
meta_a_clean = (
    meta_a_unambiguous
    .sort_values(["dup_group", "path"])
    .drop_duplicates("dup_group", keep="first")
    .reset_index(drop=True)
)

print("Animal-90 before dedupe:", len(meta_a_grouped))
print("Ambiguous cross-label duplicate groups:", len(ambiguous_dup_groups))
print("Rows removed due to label conflict:", len(ambiguous_duplicates))
print("Animal-90 after dedupe :", len(meta_a_clean))
print("Total removed         :", len(meta_a_grouped) - len(meta_a_clean))
print("Classes               :", meta_a_clean["species"].nunique())

if len(ambiguous_duplicates):
    display(
        ambiguous_duplicates[
            ["dup_group", "species", "path", "sha256", "dhash", "phash"]
        ].sort_values(["dup_group", "species"]).head(30)
    )

class_counts_clean = meta_a_clean["species"].value_counts()
display(class_counts_clean.describe().to_frame("images_per_class"))

### QUICK_MODE sampling sau khi deduplicate

In [ ]:
classes_a = set(meta_a_clean["species"].unique())
classes_b = set(meta_b["species"].unique())
overlap_classes = sorted(classes_a & classes_b)

def make_quick_subset(df, priority_classes, n_classes=20, max_per_class=40):
    all_classes = sorted(df["species"].unique())
    chosen = list(priority_classes)
    chosen += [c for c in all_classes if c not in chosen]
    chosen = chosen[:min(n_classes, len(chosen))]

    parts = []
    for c in chosen:
        sub = df[df["species"] == c]
        if len(sub) > max_per_class:
            sub = sub.sample(max_per_class, random_state=SEED)
        parts.append(sub)

    return pd.concat(parts, ignore_index=True)

if QUICK_MODE:
    work_df = make_quick_subset(
        meta_a_clean,
        overlap_classes,
        QUICK_NUM_CLASSES,
        QUICK_MAX_IMAGES_PER_CLASS,
    )
else:
    work_df = meta_a_clean.copy()

print("Working images :", len(work_df))
print("Working classes:", work_df["species"].nunique())
print("Overlap with Wild Animals:",
      sorted(set(work_df["species"]) & set(meta_b["species"])))

## 8. Fixed stratified Train / Validation / Test split

In [ ]:
label_names = sorted(work_df["species"].unique())
class_to_idx = {c: i for i, c in enumerate(label_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

work_df = work_df.copy()
work_df["label"] = work_df["species"].map(class_to_idx)

train_df, temp_df = train_test_split(
    work_df,
    test_size=0.30,
    random_state=SEED,
    stratify=work_df["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Integrity assertions: dup_group phải disjoint.
assert set(train_df["dup_group"]).isdisjoint(set(val_df["dup_group"]))
assert set(train_df["dup_group"]).isdisjoint(set(test_df["dup_group"]))
assert set(val_df["dup_group"]).isdisjoint(set(test_df["dup_group"]))

# Exact hashes cũng phải disjoint.
assert set(train_df["sha256"]).isdisjoint(set(val_df["sha256"]))
assert set(train_df["sha256"]).isdisjoint(set(test_df["sha256"]))
assert set(val_df["sha256"]).isdisjoint(set(test_df["sha256"]))

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))
print("✅ Fixed clean split created")

split_df = pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="val"),
    test_df.assign(split="test"),
], ignore_index=True)

split_df.to_csv(OUTPUT_DIR / "fixed_clean_split.csv", index=False)

## 9. Kiểm tra lại leakage sau split

In [ ]:
def perceptual_cross_split_matches(
    left_df, right_df,
    dhash_threshold=DHASH_THRESHOLD,
    phash_threshold=PHASH_THRESHOLD
):
    matches = []

    # Chỉ so trong cùng species để tránh false positive thị giác.
    common_species = sorted(set(left_df["species"]) & set(right_df["species"]))

    for species in common_species:
        left = left_df[left_df["species"] == species]
        right = right_df[right_df["species"] == species]

        for _, a in left.iterrows():
            for _, b in right.iterrows():
                dh = hamming_hex(a["dhash"], b["dhash"])
                if dh > dhash_threshold:
                    continue
                ph = hamming_hex(a["phash"], b["phash"])
                if ph <= phash_threshold:
                    matches.append({
                        "species": species,
                        "left_path": a["path"],
                        "right_path": b["path"],
                        "dhash_distance": dh,
                        "phash_distance": ph,
                    })

    return pd.DataFrame(matches)

leak_train_val = perceptual_cross_split_matches(train_df, val_df)
leak_train_test = perceptual_cross_split_matches(train_df, test_df)
leak_val_test = perceptual_cross_split_matches(val_df, test_df)

print("Train ↔ Val candidates :", len(leak_train_val))
print("Train ↔ Test candidates:", len(leak_train_test))
print("Val ↔ Test candidates  :", len(leak_val_test))

assert len(leak_train_val) == 0
assert len(leak_train_test) == 0
assert len(leak_val_test) == 0

print("✅ No exact/perceptual leakage detected after clean split")

## 10. Retrieval metrics dùng chung

In [ ]:
def topk_search(query_embedding, gallery_embeddings, k=5):
    scores = gallery_embeddings @ query_embedding
    idx = np.argsort(scores)[::-1][:k]
    return idx, scores[idx]

def precision_at_k(
    query_labels, query_embeddings,
    gallery_labels, gallery_embeddings,
    k=5
):
    vals = []
    for q_label, q_emb in zip(query_labels, query_embeddings):
        idx, _ = topk_search(q_emb, gallery_embeddings, k=k)
        vals.append(np.mean(gallery_labels[idx] == q_label))
    return float(np.mean(vals))

def average_precision_at_k(
    q_label, q_emb,
    gallery_labels, gallery_embeddings,
    k=10
):
    idx, _ = topk_search(q_emb, gallery_embeddings, k=k)
    relevant = (gallery_labels[idx] == q_label).astype(np.float32)

    if relevant.sum() == 0:
        return 0.0

    precision_sum = 0.0
    rel_seen = 0
    for rank, rel in enumerate(relevant, start=1):
        if rel:
            rel_seen += 1
            precision_sum += rel_seen / rank

    denom = min(int(np.sum(gallery_labels == q_label)), k)
    return float(precision_sum / max(1, denom))

def mean_ap_at_k(
    query_labels, query_embeddings,
    gallery_labels, gallery_embeddings,
    k=10
):
    return float(np.mean([
        average_precision_at_k(
            q_label, q_emb,
            gallery_labels, gallery_embeddings,
            k=k
        )
        for q_label, q_emb in zip(query_labels, query_embeddings)
    ]))

## 11. Model A — PCA + KNN + PCA retrieval

In [ ]:
def load_flat_image(path, size=64):
    img = Image.open(path).convert("RGB").resize((size, size))
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr.reshape(-1)

def flat_features(df):
    X = np.stack([
        load_flat_image(p, IMG_SIZE_BASELINE)
        for p in tqdm(df["path"], desc="Flat pixels")
    ])
    y = df["label"].to_numpy()
    return X, y

X_train_flat, y_train = flat_features(train_df)
X_val_flat, y_val = flat_features(val_df)
X_test_flat, y_test = flat_features(test_df)

n_components = min(
    256 if not QUICK_MODE else 128,
    X_train_flat.shape[0] - 1,
    X_train_flat.shape[1],
)

pca = PCA(n_components=n_components, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_flat)
X_val_pca = pca.transform(X_val_flat)
X_test_pca = pca.transform(X_test_flat)

knn = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    metric="cosine"
)
knn.fit(X_train_pca, y_train)

pred_test_pca = knn.predict(X_test_pca)

pca_acc = accuracy_score(y_test, pred_test_pca)
pca_f1 = f1_score(y_test, pred_test_pca, average="macro")

X_train_pca_norm = normalize(X_train_pca)
X_test_pca_norm = normalize(X_test_pca)

pca_p5 = precision_at_k(
    y_test, X_test_pca_norm,
    y_train, X_train_pca_norm,
    k=5
)
pca_map10 = mean_ap_at_k(
    y_test, X_test_pca_norm,
    y_train, X_train_pca_norm,
    k=10
)

print(f"PCA+KNN Acc      : {pca_acc:.4f}")
print(f"PCA+KNN Macro-F1 : {pca_f1:.4f}")
print(f"PCA Retrieval P@5: {pca_p5:.4f}")
print(f"PCA Retrieval mAP@10: {pca_map10:.4f}")

## 12. Model B — HOG + Linear SVM + HOG retrieval

In [ ]:
def extract_hog_features(df):
    feats = []
    labels = []

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc="HOG"
    ):
        img = (
            Image.open(row["path"])
            .convert("L")
            .resize((IMG_SIZE_HOG, IMG_SIZE_HOG))
        )
        arr = np.asarray(img)

        feat = hog(
            arr,
            orientations=9,
            pixels_per_cell=(16, 16),
            cells_per_block=(2, 2),
            block_norm="L2-Hys",
            visualize=False,
        )
        feats.append(feat)
        labels.append(row["label"])

    return np.asarray(feats), np.asarray(labels)

X_train_hog, y_train_hog = extract_hog_features(train_df)
X_val_hog, y_val_hog = extract_hog_features(val_df)
X_test_hog, y_test_hog = extract_hog_features(test_df)

# Scaling + LinearSVC cho classification.
hog_clf = make_pipeline(
    StandardScaler(),
    LinearSVC(C=1.0, random_state=SEED, max_iter=5000)
)
hog_clf.fit(X_train_hog, y_train_hog)
pred_test_hog = hog_clf.predict(X_test_hog)

hog_acc = accuracy_score(y_test_hog, pred_test_hog)
hog_f1 = f1_score(y_test_hog, pred_test_hog, average="macro")

X_train_hog_norm = normalize(X_train_hog)
X_test_hog_norm = normalize(X_test_hog)

hog_p5 = precision_at_k(
    y_test_hog, X_test_hog_norm,
    y_train_hog, X_train_hog_norm,
    k=5
)
hog_map10 = mean_ap_at_k(
    y_test_hog, X_test_hog_norm,
    y_train_hog, X_train_hog_norm,
    k=10
)

print(f"HOG+SVM Acc      : {hog_acc:.4f}")
print(f"HOG+SVM Macro-F1 : {hog_f1:.4f}")
print(f"HOG Retrieval P@5: {hog_p5:.4f}")
print(f"HOG Retrieval mAP@10: {hog_map10:.4f}")

## 13. PyTorch Dataset & transforms

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE_CNN, IMG_SIZE_CNN)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE_CNN, IMG_SIZE_CNN)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

class DataFrameImageDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        img = self.transform(img)
        return img, int(row["label"]), row["path"]

def make_loader(df, transform, shuffle=False):
    return DataLoader(
        DataFrameImageDataset(df, transform),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
    )

train_loader = make_loader(train_df, train_tf, shuffle=True)
val_loader = make_loader(val_df, eval_tf)
test_loader = make_loader(test_df, eval_tf)

## 14. Model C — ResNet18 pretrained + fine-tuning

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = nn.Linear(resnet.fc.in_features, len(label_names))
resnet = resnet.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    resnet.parameters(),
    lr=LR,
    weight_decay=1e-4
)

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    ys, preds = [], []

    for x, y, _ in tqdm(loader, leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        if is_train:
            loss.backward()
            optimizer.step()

        pred = logits.argmax(dim=1)

        total_loss += loss.item() * x.size(0)
        ys.extend(y.detach().cpu().numpy())
        preds.extend(pred.detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader.dataset),
        "acc": accuracy_score(ys, preds),
        "f1": f1_score(ys, preds, average="macro"),
    }

history = []
best_val = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(resnet, train_loader, optimizer)
    va = run_epoch(resnet, val_loader)

    history.append({
        "epoch": epoch,
        "train_loss": tr["loss"],
        "train_acc": tr["acc"],
        "val_loss": va["loss"],
        "val_acc": va["acc"],
    })

    print(
        f"Epoch {epoch:02d} | "
        f"train acc={tr['acc']:.4f} | "
        f"val acc={va['acc']:.4f}"
    )

    if va["acc"] > best_val:
        best_val = va["acc"]
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in resnet.state_dict().items()
        }

resnet.load_state_dict(best_state)
history_df = pd.DataFrame(history)
display(history_df)

### ResNet classification + confidence

In [ ]:
@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    ys, preds, confs, paths = [], [], [], []

    for x, y, p in tqdm(loader, leave=False):
        x = x.to(DEVICE)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        conf, pred = probs.max(dim=1)

        ys.extend(y.numpy())
        preds.extend(pred.cpu().numpy())
        confs.extend(conf.cpu().numpy())
        paths.extend(p)

    return np.array(ys), np.array(preds), np.array(confs), paths

y_true_resnet, y_pred_resnet, y_conf_resnet, test_paths = predict_loader(
    resnet, test_loader
)

resnet_acc = accuracy_score(y_true_resnet, y_pred_resnet)
resnet_f1 = f1_score(y_true_resnet, y_pred_resnet, average="macro")

pred_df = pd.DataFrame({
    "path": test_paths,
    "y_true": y_true_resnet,
    "y_pred": y_pred_resnet,
    "confidence": y_conf_resnet,
})
pred_df["true_species"] = pred_df["y_true"].map(idx_to_class)
pred_df["pred_species"] = pred_df["y_pred"].map(idx_to_class)

print(f"ResNet18 Acc      : {resnet_acc:.4f}")
print(f"ResNet18 Macro-F1 : {resnet_f1:.4f}")
display(pred_df.head())

### ResNet 512-D embedding retrieval

In [ ]:
resnet_fc = resnet.fc
resnet.fc = nn.Identity()
resnet.eval()

@torch.no_grad()
def extract_resnet_embeddings(df):
    loader = make_loader(df, eval_tf)
    zs, ys, paths = [], [], []

    for x, y, p in tqdm(loader, desc="ResNet embeddings", leave=False):
        x = x.to(DEVICE)
        z = resnet(x)
        z = torch.nn.functional.normalize(z, dim=1)

        zs.append(z.cpu().numpy())
        ys.extend(y.numpy())
        paths.extend(p)

    return np.concatenate(zs), np.asarray(ys), paths

gallery_resnet, gallery_labels_resnet, gallery_paths_resnet =     extract_resnet_embeddings(train_df)
query_resnet, query_labels_resnet, query_paths_resnet =     extract_resnet_embeddings(test_df)

resnet.fc = resnet_fc

resnet_p5 = precision_at_k(
    query_labels_resnet, query_resnet,
    gallery_labels_resnet, gallery_resnet,
    k=5
)
resnet_map10 = mean_ap_at_k(
    query_labels_resnet, query_resnet,
    gallery_labels_resnet, gallery_resnet,
    k=10
)

print(f"ResNet Retrieval P@5   : {resnet_p5:.4f}")
print(f"ResNet Retrieval mAP@10: {resnet_map10:.4f}")

## 15. Model D — DINOv2 frozen embeddings + Linear SVM / FAISS

In [ ]:
print("Loading DINOv2 ViT-S/14...")
dinov2 = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14"
)
dinov2 = dinov2.to(DEVICE).eval()

@torch.no_grad()
def extract_dino_embeddings(df):
    loader = make_loader(df, eval_tf)
    zs, ys, paths = [], [], []

    for x, y, p in tqdm(loader, desc="DINOv2 embeddings", leave=False):
        x = x.to(DEVICE)
        z = dinov2(x)
        z = torch.nn.functional.normalize(z, dim=1)

        zs.append(z.cpu().numpy())
        ys.extend(y.numpy())
        paths.extend(p)

    return np.concatenate(zs), np.asarray(ys), paths

gallery_dino, gallery_labels_dino, gallery_paths_dino =     extract_dino_embeddings(train_df)
val_dino, val_labels_dino, _ =     extract_dino_embeddings(val_df)
query_dino, query_labels_dino, query_paths_dino =     extract_dino_embeddings(test_df)

print("Train DINO:", gallery_dino.shape)
print("Test DINO :", query_dino.shape)

### DINOv2 classification

In [ ]:
dino_clf = LinearSVC(
    C=1.0,
    random_state=SEED,
    max_iter=5000
)
dino_clf.fit(gallery_dino, gallery_labels_dino)

pred_test_dino = dino_clf.predict(query_dino)

dino_acc = accuracy_score(query_labels_dino, pred_test_dino)
dino_f1 = f1_score(query_labels_dino, pred_test_dino, average="macro")

print(f"DINOv2 + Linear SVM Acc     : {dino_acc:.4f}")
print(f"DINOv2 + Linear SVM Macro-F1: {dino_f1:.4f}")

### DINOv2 retrieval bằng FAISS exact cosine search

In [ ]:
d = gallery_dino.shape[1]
faiss_index = faiss.IndexFlatIP(d)
faiss_index.add(gallery_dino.astype("float32"))

# Metric dùng chung để so sánh trực tiếp.
dino_p5 = precision_at_k(
    query_labels_dino, query_dino,
    gallery_labels_dino, gallery_dino,
    k=5
)
dino_map10 = mean_ap_at_k(
    query_labels_dino, query_dino,
    gallery_labels_dino, gallery_dino,
    k=10
)

print(f"DINOv2 Retrieval P@5   : {dino_p5:.4f}")
print(f"DINOv2 Retrieval mAP@10: {dino_map10:.4f}")

## 16. Cross-dataset integrity audit: Animal-90 ↔ Wild Animals

Đây là bước bắt buộc trước khi diễn giải cross-dataset performance.

External query bị loại nếu:
- SHA256 trùng bất kỳ ảnh Animal-90; hoặc
- cùng species và đồng thời dHash/pHash nằm trong ngưỡng near-duplicate.

Ta so với **toàn bộ Animal-90 clean source**, không chỉ Gallery Train, để benchmark external bảo thủ hơn.

In [ ]:
def filter_cross_dataset_duplicates(
    external_df,
    reference_df,
    dhash_threshold=DHASH_THRESHOLD,
    phash_threshold=PHASH_THRESHOLD,
):
    ref_sha = set(reference_df["sha256"])
    ref_by_species = {
        s: part[["dhash", "phash", "path"]].to_dict("records")
        for s, part in reference_df.groupby("species")
    }

    keep = []
    removed = []

    for row in tqdm(
        external_df.to_dict("records"),
        desc="Cross-dataset duplicate audit"
    ):
        reason = None

        if row["sha256"] in ref_sha:
            reason = "exact_sha256"
        else:
            refs = ref_by_species.get(row["species"], [])
            for r in refs:
                dh = hamming_hex(row["dhash"], r["dhash"])
                if dh > dhash_threshold:
                    continue
                ph = hamming_hex(row["phash"], r["phash"])
                if ph <= phash_threshold:
                    reason = f"near_duplicate(dh={dh},ph={ph})"
                    break

        if reason is None:
            keep.append(row)
        else:
            removed.append({**row, "removal_reason": reason})

    return pd.DataFrame(keep), pd.DataFrame(removed)

known_external = meta_b[
    meta_b["species"].isin(class_to_idx)
].copy()

unknown_external = meta_b[
    ~meta_b["species"].isin(class_to_idx)
].copy()

known_external_clean, cross_removed = filter_cross_dataset_duplicates(
    known_external,
    meta_a_clean
)

known_external_clean["label"] =     known_external_clean["species"].map(class_to_idx)

print("Known external before audit:", len(known_external))
print("Removed as cross-dataset duplicate candidates:", len(cross_removed))
print("Known external after audit :", len(known_external_clean))
print("Unknown/OOD candidates     :", len(unknown_external))
print("Known external species     :",
      sorted(known_external_clean["species"].unique()))

if len(cross_removed):
    display(
        cross_removed[
            ["species", "path", "removal_reason"]
        ].head(20)
    )

### External sampling cho QUICK_MODE

In [ ]:
if QUICK_MODE:
    parts = []
    for species, part in known_external_clean.groupby("species"):
        parts.append(
            part.sample(
                min(len(part), 50),
                random_state=SEED
            )
        )
    external_eval = (
        pd.concat(parts, ignore_index=True)
        if parts else known_external_clean.copy()
    )
else:
    external_eval = known_external_clean.copy()

print("External evaluation queries:", len(external_eval))

## 17. Cross-dataset ResNet & DINOv2 retrieval

In [ ]:
if len(external_eval):
    # ResNet external embedding.
    resnet.fc = nn.Identity()
    external_resnet, external_labels_resnet, external_paths =         extract_resnet_embeddings(external_eval)
    resnet.fc = resnet_fc

    resnet_cross_p5 = precision_at_k(
        external_labels_resnet, external_resnet,
        gallery_labels_resnet, gallery_resnet,
        k=5
    )
    resnet_cross_map10 = mean_ap_at_k(
        external_labels_resnet, external_resnet,
        gallery_labels_resnet, gallery_resnet,
        k=10
    )

    # DINOv2 external embedding.
    external_dino, external_labels_dino, _ =         extract_dino_embeddings(external_eval)

    dino_cross_p5 = precision_at_k(
        external_labels_dino, external_dino,
        gallery_labels_dino, gallery_dino,
        k=5
    )
    dino_cross_map10 = mean_ap_at_k(
        external_labels_dino, external_dino,
        gallery_labels_dino, gallery_dino,
        k=10
    )
else:
    resnet_cross_p5 = np.nan
    resnet_cross_map10 = np.nan
    dino_cross_p5 = np.nan
    dino_cross_map10 = np.nan

print(
    f"ResNet cross P@5={resnet_cross_p5:.4f} | "
    f"mAP@10={resnet_cross_map10:.4f}"
)
print(
    f"DINOv2 cross P@5={dino_cross_p5:.4f} | "
    f"mAP@10={dino_cross_map10:.4f}"
)

## 18. Benchmark Dashboard — cùng split, cùng metric

In [ ]:
results_df = pd.DataFrame([
    {
        "model": "PCA + KNN",
        "classification_acc": pca_acc,
        "classification_macro_f1": pca_f1,
        "retrieval_p@5": pca_p5,
        "retrieval_map@10": pca_map10,
        "cross_dataset_p@5": np.nan,
        "cross_dataset_map@10": np.nan,
    },
    {
        "model": "HOG + Linear SVM",
        "classification_acc": hog_acc,
        "classification_macro_f1": hog_f1,
        "retrieval_p@5": hog_p5,
        "retrieval_map@10": hog_map10,
        "cross_dataset_p@5": np.nan,
        "cross_dataset_map@10": np.nan,
    },
    {
        "model": "ResNet18 fine-tuned",
        "classification_acc": resnet_acc,
        "classification_macro_f1": resnet_f1,
        "retrieval_p@5": resnet_p5,
        "retrieval_map@10": resnet_map10,
        "cross_dataset_p@5": resnet_cross_p5,
        "cross_dataset_map@10": resnet_cross_map10,
    },
    {
        "model": "DINOv2 frozen + Linear SVM / FAISS",
        "classification_acc": dino_acc,
        "classification_macro_f1": dino_f1,
        "retrieval_p@5": dino_p5,
        "retrieval_map@10": dino_map10,
        "cross_dataset_p@5": dino_cross_p5,
        "cross_dataset_map@10": dino_cross_map10,
    },
])

display(
    results_df.style.format({
        c: "{:.4f}"
        for c in results_df.columns
        if c != "model"
    })
)

### Biểu đồ so sánh

In [ ]:
plot_df = results_df.set_index("model")

plt.figure(figsize=(11, 5))
plot_df[
    ["classification_acc", "classification_macro_f1"]
].plot(kind="bar", ax=plt.gca())
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Clean Benchmark v1 — Classification")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
plot_df[
    ["retrieval_p@5", "retrieval_map@10"]
].plot(kind="bar", ax=plt.gca())
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Clean Benchmark v1 — In-domain Retrieval")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 19. Error analysis: high-confidence wrong predictions của ResNet

In [ ]:
resnet_errors = pred_df[
    pred_df["y_true"] != pred_df["y_pred"]
].copy()

print("ResNet errors:", len(resnet_errors))

if len(resnet_errors):
    display(
        resnet_errors
        .sort_values("confidence", ascending=False)
        [["true_species", "pred_species", "confidence", "path"]]
        .head(20)
    )

    confusion_pairs = (
        resnet_errors
        .groupby(["true_species", "pred_species"])
        .size()
        .sort_values(ascending=False)
        .rename("count")
        .reset_index()
    )
    display(confusion_pairs.head(20))

## 20. Data integrity summary

In [ ]:
integrity_summary = pd.DataFrame([
    {
        "item": "Animal-90 raw image rows",
        "value": len(meta_a_raw),
    },
    {
        "item": "Animal-90 corrupted/unreadable",
        "value": len(bad_a),
    },
    {
        "item": "Animal-90 valid before dedupe",
        "value": len(meta_a_grouped),
    },
    {
        "item": "Animal-90 clean representatives",
        "value": len(meta_a_clean),
    },
    {
        "item": "Animal-90 removed duplicate/near-duplicate rows",
        "value": len(meta_a_grouped) - len(meta_a_clean),
    },
    {
        "item": "Animal-90 ambiguous cross-label duplicate groups",
        "value": len(ambiguous_dup_groups),
    },
    {
        "item": "Working classes",
        "value": work_df["species"].nunique(),
    },
    {
        "item": "Train ↔ Val leakage after clean split",
        "value": len(leak_train_val),
    },
    {
        "item": "Train ↔ Test leakage after clean split",
        "value": len(leak_train_test),
    },
    {
        "item": "Wild external cross-dataset duplicates removed",
        "value": len(cross_removed),
    },
    {
        "item": "Wild external valid evaluation queries",
        "value": len(external_eval),
    },
])

display(integrity_summary)

## 21. Lưu toàn bộ artifact của benchmark

In [ ]:
results_df.to_csv(
    OUTPUT_DIR / "benchmark_results.csv",
    index=False
)
integrity_summary.to_csv(
    OUTPUT_DIR / "integrity_summary.csv",
    index=False
)
pred_df.to_csv(
    OUTPUT_DIR / "resnet_test_predictions.csv",
    index=False
)
meta_a_clean.to_csv(
    OUTPUT_DIR / "animal90_clean_metadata.csv",
    index=False
)
known_external_clean.to_csv(
    OUTPUT_DIR / "wild_animals_clean_external_metadata.csv",
    index=False
)

torch.save({
    "model_state_dict": resnet.state_dict(),
    "class_to_idx": class_to_idx,
    "idx_to_class": idx_to_class,
    "quick_mode": QUICK_MODE,
    "seed": SEED,
}, OUTPUT_DIR / "resnet18_clean_v1.pt")

np.save(
    OUTPUT_DIR / "dino_gallery_embeddings.npy",
    gallery_dino
)
np.save(
    OUTPUT_DIR / "dino_gallery_labels.npy",
    gallery_labels_dino
)

print("Saved benchmark artifacts to:", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)

# ✅ Clean Benchmark v1 hoàn thành

Sau khi notebook chạy xong, chỉ sử dụng **Dashboard ở Phase 18** cho báo cáo benchmark.

## Cách chạy chính thức

1. Đảm bảo Colab đang dùng GPU.
2. Đổi:
   ```python
   QUICK_MODE = False
   ```
3. `Runtime → Restart session`
4. `Run all`
5. Lưu:
   - `benchmark_results.csv`
   - `integrity_summary.csv`
   - `fixed_clean_split.csv`

## Sau Benchmark v1

Khi full 90-class clean benchmark đã ổn, roadmap hợp lý là:

1. ResNet18 from scratch vs frozen vs fine-tuned.
2. Confidence calibration: ECE + Temperature Scaling.
3. OOD/Open-set experiment với species Wild-only (ví dụ cheetah nếu không có trong Animal-90).
4. SigLIP2 comparison.
5. Grad-CAM + retrieval failure taxonomy.
6. Gradio/Streamlit demo.